# 05. Hotspot Analysis and GEE Extraction

This notebook provides an interactive way to inspect the results of the HPC hotspot extraction script (`scripts/run_hotspot_ensemble.py`). It visualizes the high-uncertainty hotspots identified by the Dask pipeline and verifies that the Google Earth Engine (GEE) true-color composites have been successfully queued for export.

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from pathlib import Path
import numpy as np
import ee
import geemap
import logging

# Setup logging
logging.basicConfig(level=logging.INFO)

# Initialize GEE (Requires gee_config.yaml or GEE_PROJECT_ID)
from wetland_analysis.data.config import load_gee_config
config = load_gee_config()
if config.get('gee_project_id') == 'your-gee-project-id':
    print("WARNING: Using default dummy project ID. GEE validation will be skipped.")
else:
    try:
        ee.Initialize(project=config.get('gee_project_id'))
        print("GEE Initialized.")
    except Exception as e:
        print(f"Failed to initialize GEE: {e}")

## Load Results from Zarr
Ensure you have run the `scripts/run_hotspot_ensemble.py` script first to generate the `results/hotspots_ensemble.zarr` file.

In [ ]:
zarr_path = Path("../results/hotspots_ensemble.zarr")
if zarr_path.exists():
    ds = xr.open_zarr(zarr_path, consolidated=True)
    print(ds)
else:
    print(f"{zarr_path} not found. Please run the HPC script first.")
    ds = None

## Visualize Uncertainty Map
Plot the Shannon Entropy map.

In [ ]:
if ds is not None and 'shannon_entropy' in ds:
    fig, ax = plt.subplots(figsize=(10, 8), subplot_kw={'projection': ccrs.PlateCarree()})
    
    # Plot the entropy
    ds['shannon_entropy'].plot(ax=ax, transform=ccrs.PlateCarree(), cmap='viridis', robust=True)
    ax.coastlines()
    ax.set_title("Wetland Ensemble Uncertainty (Shannon Entropy)")
    plt.show()